In [ ]:
!git clone https://github.com/RomanShamailov/codebert_tta_detection.git

%cd codebert_tta_detection

In [ ]:
!unzip -q gptsniffer_finetuning/dataset.zip -d gptsniffer_finetuning
!find gptsniffer_finetuning/dataset -maxdepth 2 -type f | sort | head

In [ ]:
!pip install -r requirements.txt

In [ ]:
import wandb

wandb.login(key="YOUR_KEY")

In [ ]:
DATASETS = {
    "python": "gptsniffer_python_test",
    "python_no_comment": "gptsniffer_python_no_comment_test",
    "java": "gptsniffer_java_test",
    "aigcodeset": "gptsniffer_aigcodeset_test",
}

TTA_METHODS = {
    "none": ["tta=none"],
    "tent": ["tta=tent", "tta.reset_each_batch=true"],
    "tent_accumulation": ["tta=tent", "tta.reset_each_batch=false"],
    "centroids": ["tta=centroids"],
}

PROJECT_NAME = "gptsniffer-tta-detection"

In [ ]:
import os
from IPython import get_ipython

os.environ["HYDRA_FULL_ERROR"] = "1"

ipython = get_ipython()

for dataset_alias, dataset_config in DATASETS.items():
    for method_alias, method_overrides in TTA_METHODS.items():
        run_name = f"{dataset_alias}_{method_alias}"
        command = " ".join([
            "python -u inference.py",
            f"datasets={dataset_config}",
            "writer=wandb",
            f"writer.project_name={PROJECT_NAME}",
            f"writer.run_name={run_name}",
            f"inferencer.save_path={run_name}",
            *method_overrides,
        ])

        print("Running:", command, flush=True)
        result = ipython.system(command)

        if result not in (None, 0):
            raise RuntimeError(f"Command failed: {command}")